In [1]:
import sys

sys.path.append("../")
from agents.trainers.ppo_trainer import PPOTrainer
import gymnasium as gym
import torch
from configs.agents.ppo import PPOConfig
from game_configs.cartpole_config import CartPoleConfig
from stats.stats import StatTracker

env = gym.make("CartPole-v1", render_mode="rgb_array")
config_dict = {
    # ... general params ...
    "activation": "tanh",
    "clip_param": 0.2,
    "kernel_initializer": "orthogonal",
    "discount_factor": 0.99,
    "gae_lambda": 0.95,
    "steps_per_epoch": 512,
    "train_policy_iterations": 4,
    "train_value_iterations": 4,
    "target_kl": 0.02,
    "entropy_coefficient": 0.01,
    "num_minibatches": 4,
    "loss_function": None,  # Should be handled by new strategies if applicable
    "training_steps": 500,
    # --- New Modular Backbones ---
    # --- New Modular Heads (replacing dense_layers) ---
    "policy_head": {
        "neck": {
            "type": "dense",
            "widths": [64],
        },  # Or add layers here if actor_dense_layers meant head layers
        # "output_strategy": ... (defaults to Categorical if not set)
    },
    "value_head": {
        "neck": {
            "type": "dense",
            "widths": [64],
        },  # Or add layers here
        # "output_strategy": ... (defaults to Regression if not set)
    },
    "actor_config": {
        "learning_rate": 2.5e-4,
        "adam_epsilon": 1e-7,
        "clipnorm": 0.5,
    },
    "critic_config": {
        "learning_rate": 2.5e-4,
        "adam_epsilon": 1e-7,
        "clipnorm": 0.5,
    },
}
print("PPO Config")
config = PPOConfig(
    config_dict,
    CartPoleConfig(),
)

trainer = PPOTrainer(
    config,
    env,
    torch.device("cpu"),
    StatTracker("ppo"),
)

trainer.checkpoint_interval = 100
trainer.test_interval = 100
trainer.test_trials = 50

trainer.train()

PPO Config
Using default model_name                    : agent
Using         training_steps                : 500
Using default adam_epsilon                  : 1e-08
Using default momentum                      : 0.9
Using default learning_rate                 : 0.001
Using default clipnorm                      : 0
Using default optimizer                     : <class 'torch.optim.adam.Adam'>
Using default weight_decay                  : 0.0
Using         num_minibatches               : 4
Using default training_iterations           : 1
Using default lr_schedule_type              : none
Using default lr_schedule_steps             : []
Using default lr_schedule_values            : []
Using default minibatch_size                : 64
Using default replay_buffer_size            : 5000
Using default min_replay_buffer_size        : 64
Using default n_step                        : 1
Using         discount_factor               : 0.99
Using default per_alpha                     : 0.5
Using default 

In [ ]:
import gymnasium as gym
import sys

sys.path.append("../")
from agents.trainers.rainbow_trainer import RainbowTrainer
from game_configs.cartpole_config import CartPoleConfig
from configs.agents.rainbow_dqn import RainbowConfig
from losses.basic_losses import KLDivergenceLoss
from stats.stats import StatTracker
import torch

# env = ClipReward(AtariPreprocessing(gym.make("MsPacmanNoFrameskip-v4", render_mode="rgb_array"), terminal_on_life_loss=True), -1, 1) # as recommended by the original paper, should already include max pooling
# env = FrameStack(env, 4)
env = gym.make("CartPole-v1")

config_dict = {
    # --- General Params ---
    "adam_epsilon": 1e-8,
    "learning_rate": 0.001,
    "training_steps": 5000,
    "minibatch_size": 128,
    "transfer_interval": 100,
    "n_step": 3,
    "replay_interval": 1,
    "kernel_initializer": "orthogonal",
    "clipnorm": 10.0,
    "model_name": "rainbow_refactor",
    # --- Distributional / C51 Strategy ---
    # Global noisy_sigma for the architecture
    "noisy_sigma": 0.5,
    # --- Backbones ---
    "backbone": {
        "type": "dense",
        "widths": [128],
    },
    # --- Head Configuration (New Nested System) ---
    "head": {
        "output_strategy": {
            "type": "c51",
            "num_atoms": 51,
            # Defaults for v_min/v_max are handled automatically
            # or you can specify them here:
            "v_min": 0,
            "v_max": 500,
        },
        # Hidden layers for Dueling Head (restored to 512 for performance)
        "value_hidden_widths": [],
        "advantage_hidden_widths": [],
        # "noisy_sigma": 0.5 (inherits from global)
    },
    # --- PER (Prioritized Experience Replay) ---
    "per_epsilon": 1e-6,
    "per_alpha": 0.2,
    "per_beta": 0.6,
}


game_config = CartPoleConfig()
config = RainbowConfig(config_dict, game_config)
trainer = RainbowTrainer(
    config, env, torch.device("cpu"), StatTracker("rainbow_refactor")
)

trainer.checkpoint_interval = 100
trainer.test_interval = 500

trainer.train()

Using         model_name                    : rainbow_refactor
Using         training_steps                : 5000
Using         adam_epsilon                  : 1e-08
Using default momentum                      : 0.9
Using         learning_rate                 : 0.001
Using         clipnorm                      : 10.0
Using default optimizer                     : <class 'torch.optim.adam.Adam'>
Using default weight_decay                  : 0.0
Using default num_minibatches               : 1
Using default training_iterations           : 1
Using default lr_schedule_type              : none
Using default lr_schedule_steps             : []
Using default lr_schedule_values            : []
Using         minibatch_size                : 128
Using default replay_buffer_size            : 5000
Using default min_replay_buffer_size        : 128
Using         n_step                        : 3
Using default discount_factor               : 0.99
Using         per_alpha                     : 0.2
Using   

KeyboardInterrupt: 

In [ ]:
import sys

sys.path.append("../")
import time
import torch
from losses.basic_losses import CategoricalCrossentropyLoss, KLDivergenceLoss
from agents.random import RandomAgent

# from agents.muzero import MuZeroAgent
from agents.trainers.muzero_trainer import MuZeroTrainer
from configs.agents.muzero import MuZeroConfig
from game_configs.tictactoe_config import TicTacToeConfig
from agents.tictactoe_expert import TicTacToeBestAgent
from modules.world_models.muzero_world_model import MuzeroWorldModel
from stats.stats import StatTracker

# Ensure we use CPU for fairness/comparibility or GPU if available
device = "cpu"  # or "cuda" if available
print(f"Using device: {device}")

params = {
    "num_simulations": 25,
    "per_alpha": 0.0,
    "per_beta": 0.0,
    "per_beta_final": 0.0,
    "n_step": 10,
    "root_dirichlet_alpha": 0.25,
    # Architecture Config (Shared settings)
    "architecture": {
        "residual_layers": [(24, 3, 1)],
    },
    # Main Backbone (Representation/Dynamics)
    "backbone": {
        "type": "resnet",
        "filters": [24],
        "kernel_sizes": [3],
        "strides": [1],
    },
    # Specialized Heads
    "reward_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
        },
        "output_strategy": {"type": "regression"},
    },
    "value_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
        },
        "output_strategy": {"type": "regression"},
    },
    "policy_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
        }
    },
    "to_play_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
        }
    },
    # Legacy fields that are still parsed for now
    "known_bounds": [-1, 1],
    "minibatch_size": 8,
    "replay_buffer_size": 100000,
    "gumbel": False,
    "gumbel_m": 16,
    "policy_loss_function": CategoricalCrossentropyLoss(),
    "training_steps": 10000,
    "transfer_interval": 1,
    "num_workers": 4,
    "world_model_cls": MuzeroWorldModel,
    "search_batch_size": 0,
    "use_virtual_mean": False,
    "virtual_loss": 3.0,
}

game_config = TicTacToeConfig()

print("--- Running MuZero Comaprison of Changes ---")
params_batched = params.copy()
params_batched["search_batch_size"] = 5
params_batched["use_virtual_mean"] = True
params_batched["model_name"] = "muzero_search_refactor"

env_batch = TicTacToeConfig().make_env()
config_batch = MuZeroConfig(config_dict=params_batched, game_config=game_config)
config_batch.search_batch_size = 5  # Explicitly set

trainer = MuZeroTrainer(
    config_batch,
    env_batch,
    torch.device("cpu"),
    stats=StatTracker(model_name="muzero_search_refactor"),
    test_agents=[RandomAgent(), TicTacToeBestAgent()],
)
trainer.checkpoint_interval = 100
trainer.test_interval = 1000
trainer.test_trials = 100

start_time = time.time()
trainer.train()
end_time = time.time()
print(f"MuZero Batched Time: {end_time - start_time:.2f}s")

Using device: cpu
--- Running MuZero Comaprison of Changes ---
Using         model_name                    : muzero_search_refactor
Using         training_steps                : 10000
Using default adam_epsilon                  : 1e-08
Using default momentum                      : 0.9
Using default learning_rate                 : 0.001
Using default clipnorm                      : 0
Using default optimizer                     : <class 'torch.optim.adam.Adam'>
Using default weight_decay                  : 0.0
Using default num_minibatches               : 1
Using default training_iterations           : 1
Using default lr_schedule_type              : none
Using default lr_schedule_steps             : []
Using default lr_schedule_values            : []
Using         minibatch_size                : 8
Using         replay_buffer_size            : 100000
Using default min_replay_buffer_size        : 8
Using         n_step                        : 10
Using default discount_factor              

In [1]:
import sys

sys.path.append("../")
import os
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
import random

from custom_gym_envs.envs.matching_pennies import (
    env as matching_pennies_env,
    MatchingPenniesGymEnv,
)
from agents.trainers.nfsp_trainer import NFSPTrainer
from agent_configs.dqn.nfsp_config import NFSPDQNConfig
from agent_configs.dqn.rainbow_config import RainbowConfig
from game_configs.matching_pennies_config import MatchingPenniesConfig
from losses.basic_losses import (
    KLDivergenceLoss,
    CategoricalCrossentropyLoss,
    HuberLoss,
    MSELoss,
)
import torch
from torch.optim import Adam, SGD
from stats.stats import StatTracker

config_dict = {
    "shared_networks_and_buffers": False,
    "training_steps": 10000,
    "anticipatory_param": 0.1,
    "replay_interval": 2,
    "num_minibatches": 1,
    "learning_rate": 0.1,
    "momentum": 0.0,
    "optimizer": SGD,
    "loss_function": HuberLoss(),
    "min_replay_buffer_size": 500,
    "minibatch_size": 128,
    "replay_buffer_size": 1000,
    "transfer_interval": 300,
    # --- NEW RL BACKBONE ---
    "backbone": {"type": "dense", "widths": [128]},
    # Note: value_backbone and advantage_backbone default to Identity (None)
    # which replaces the old empty list [] behavior.
    "noisy_sigma": 0.0,
    "eg_epsilon": 0.06,
    "eg_epsilon_decay_type": "inverse_sqrt",
    "eg_epsilon_decay_final_step": 0,
    # --- SL CONFIG ---
    "sl_learning_rate": 0.01,
    "sl_momentum": 0.0,
    "sl_optimizer": SGD,
    "sl_loss_function": CategoricalCrossentropyLoss(),
    "sl_min_replay_buffer_size": 500,
    "sl_minibatch_size": 128,
    "sl_replay_buffer_size": 20000,
    # --- NEW SL BACKBONE ---
    "sl_backbone": {"type": "dense", "widths": [128]},
    "sl_clip_low_prob": 0.0,
    "per_alpha": 0.0,
    "per_beta": 0.0,
    "per_beta_final": 0.0,
    "per_epsilon": 0.00001,
    "n_step": 1,
    "atom_size": 1,
    "dueling": False,
    "clipnorm": 10.0,
    "sl_clipnorm": 10.0,
}

config = NFSPDQNConfig(
    config_dict=config_dict,
    game_config=MatchingPenniesConfig(),
)
config.save_intermediate_weights = True

import custom_gym_envs
import gymnasium as gym

# env = gym.make("custom_gym_envs/LeducHoldem-v0", encode_player_turn=False)

env = matching_pennies_env(render_mode="rgb_array", max_cycles=1)

trainer = NFSPTrainer(config, env, torch.device("cpu"), StatTracker("nfsp_3"))
trainer.checkpoint_interval = 100
trainer.test_interval = 1000
trainer.test_trials = 10000
trainer.train()

NFSPDQNConfig
Using default save_intermediate_weights     : False
Using         training_steps                : 10000
Using default adam_epsilon                  : 1e-08
Using         momentum                      : 0.0
Using         learning_rate                 : 0.1
Using         clipnorm                      : 10.0
Using         optimizer                     : <class 'torch.optim.sgd.SGD'>
Using default weight_decay                  : 0.0
Using         num_minibatches               : 1
Using default training_iterations           : 1
Using default lr_schedule_type              : none
Using default lr_schedule_steps             : []
Using default lr_schedule_values            : []
Using         minibatch_size                : 128
Using         replay_buffer_size            : 1000
Using         min_replay_buffer_size        : 500
Using         n_step                        : 1
Using default discount_factor               : 0.99
Using         per_alpha                     : 0.0
Using   

/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/gymnasium/envs/registration.py:636: UserWarning: WARN: Overriding environment custom_gym_envs/Catan-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/gymnasium/envs/registration.py:636: UserWarning: WARN: Overriding environment custom_gym_envs/MatchingPennies-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/gymnasium/envs/registration.py:636: UserWarning: WARN: Overriding environment custom_gym_envs/Connect4-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/gymnasium/envs/

Initializing stat 'learner_fps' with subkeys None
Initializing stat 'player_0_rl_loss' with subkeys None
Initializing stat 'player_1_rl_loss' with subkeys None
Step 10/10000, Avg Score: 0.80, Time/10 steps: 1.01s
Step 20/10000, Avg Score: 0.70, Time/10 steps: 1.04s
Step 30/10000, Avg Score: 0.80, Time/10 steps: 1.04s
Step 40/10000, Avg Score: 0.70, Time/10 steps: 1.04s
Initializing stat 'player_1_sl_loss' with subkeys None
Initializing stat 'player_0_sl_loss' with subkeys None
Step 50/10000, Avg Score: 0.30, Time/10 steps: 1.05s
Step 60/10000, Avg Score: 0.20, Time/10 steps: 1.05s
Step 70/10000, Avg Score: 0.00, Time/10 steps: 1.05s
Step 80/10000, Avg Score: -0.10, Time/10 steps: 1.04s
Step 90/10000, Avg Score: 0.10, Time/10 steps: 1.04s
Step 100/10000, Avg Score: 0.30, Time/10 steps: 1.04s
plotting score
plotting sl_policy
plotting episode_length
plotting num_episodes
plotting actor_fps
plotting learner_fps
plotting player_0_rl_loss
plotting player_1_rl_loss
plotting player_1_sl_loss


KeyboardInterrupt: 